<a href="https://colab.research.google.com/github/YASHAS-KING/BIS_LAB/blob/main/BIS_ACO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import random
import math

class VRPEnvironment:
    def __init__(self, num_customers, capacity):
        self.num_customers = num_customers
        self.capacity = capacity
        self.depot = 0
        # Random demands between 5 and 20 for customers; Depot (index 0) has 0 demand
        self.demands = [random.randint(5, 20) for _ in range(num_customers + 1)]
        self.demands[0] = 0
        self.pheromones = {}

        # Generate random coordinates (x, y) to calculate consistent distances
        self.coordinates = {i: (random.uniform(0, 100), random.uniform(0, 100)) for i in range(num_customers + 1)}
        self.distances = {}

        # Pre-calculate a symmetric distance matrix using Euclidean distance
        for i in range(num_customers + 1):
            for j in range(num_customers + 1):
                if i != j:
                    x1, y1 = self.coordinates[i]
                    x2, y2 = self.coordinates[j]
                    self.distances[(i, j)] = math.hypot(x2 - x1, y2 - y1)
                else:
                    self.distances[(i, j)] = 0

def aco_vrp(env, num_ants, num_iterations, alpha=1.0, beta=2.0, evaporation_rate=0.1):
    best_overall_route = None
    best_overall_distance = float('inf')

    for iteration in range(num_iterations):
        for ant in range(num_ants):
            current_node = env.depot
            current_capacity = env.capacity
            visited = {env.depot}
            route = [env.depot]
            total_distance = 0

            while len(visited) < env.num_customers + 1:
                unvisited = [n for n in range(env.num_customers + 1) if n not in visited]

                # Check which unvisited nodes can be visited without exceeding vehicle capacity
                feasible_unvisited = [n for n in unvisited if env.demands[n] <= current_capacity]

                if not feasible_unvisited:
                    # Vehicle is full; must return to depot to restock/drop off
                    total_distance += get_distance(env, current_node, env.depot)
                    current_node = env.depot
                    current_capacity = env.capacity
                    route.append(env.depot)
                else:
                    # Select the next node probabilistically
                    next_node = select_next_node(env, current_node, feasible_unvisited, alpha, beta)

                    total_distance += get_distance(env, current_node, next_node)
                    current_capacity -= env.demands[next_node]
                    current_node = next_node
                    visited.add(next_node)
                    route.append(next_node)

            # Final return to the depot after all customers are visited
            if current_node != env.depot:
                total_distance += get_distance(env, current_node, env.depot)
                route.append(env.depot)

            # Optional: Apply local search (stub)
            route, total_distance = apply_local_search(route, total_distance)

            # Track the best route found across all ants and iterations
            if total_distance < best_overall_distance:
                best_overall_distance = total_distance
                best_overall_route = route

        # Update pheromones only based on the best route found (Elitist strategy)
        update_pheromones(env, evaporation_rate, best_overall_route, best_overall_distance)

    return best_overall_route, best_overall_distance

def select_next_node(env, current_node, feasible_nodes, alpha, beta):
    probabilities = []
    denominator = 0.0

    for node in feasible_nodes:
        pheromone = env.pheromones.get((current_node, node), 1.0)
        distance = get_distance(env, current_node, node)
        visibility = 1.0 / distance if distance > 0 else 0.0001

        # Calculate attractiveness based on pheromone trail and closeness (visibility)
        value = (pheromone ** alpha) * (visibility ** beta)
        probabilities.append(value)
        denominator += value

    # Roulette wheel selection based on calculated probabilities
    rand = random.random()
    cumulative = 0.0
    for i, prob in enumerate(probabilities):
        cumulative += (prob / denominator)
        if rand <= cumulative:
            return feasible_nodes[i]
    return feasible_nodes[-1]

def get_distance(env, node1, node2):
    return env.distances.get((node1, node2), float('inf'))

def apply_local_search(route, distance):
    # Placeholder for local search optimizations like 2-Opt or Or-Opt
    return route, distance

def update_pheromones(env, evaporation_rate, best_route, best_distance):
    # 1. Evaporate existing pheromones globally
    for edge in env.pheromones:
        env.pheromones[edge] *= (1.0 - evaporation_rate)

    # 2. Deposit new pheromones only on the edges of the best route
    deposit_amount = 100.0 / best_distance
    for i in range(len(best_route) - 1):
        edge = (best_route[i], best_route[i+1])
        env.pheromones[edge] = env.pheromones.get(edge, 1.0) + deposit_amount

# --- Execution Block ---
if __name__ == "__main__":
    # Initialize environment: 10 Customers, Vehicle Capacity of 40
    # Random seed fixed for reproducible output in this example
    random.seed(42)
    env = VRPEnvironment(num_customers=10, capacity=40)

    print(f"Customer Demands (Index 0 is Depot): {env.demands}")
    print("Running Ant Colony Optimization...\n")

    # Run algorithm with 20 ants over 50 iterations
    best_route, best_distance = aco_vrp(env, num_ants=20, num_iterations=50)

    print(f"Best Route Found: {best_route}")
    print(f"Total Distance Travelled: {best_distance:.2f}")

Customer Demands (Index 0 is Depot): [0, 5, 13, 12, 12, 9, 8, 7, 18, 6, 5]
Running Ant Colony Optimization...

Best Route Found: [0, 7, 4, 2, 1, 0, 10, 9, 6, 3, 5, 0, 8, 0]
Total Distance Travelled: 416.57
